# SPIDER-Seq: naturally overlapping target panels — OnDemand

Adult1, Adult2 and Adult3 share a molecular input space and have 12, 14 and 16
assayed targets (24 in the union). This is the scRNA-seq Adult.Ex object,
distinct from the spatial SPIDER notebook. The injection-panel mask is native;
no target blocks or positive labels are artificially hidden.

The primary benchmark evaluates held-out cells on targets actually assayed
in their animal. Naturally unassayed pairs receive separate, unvalidated
forecasts. There is no measured full-panel control for those absent labels.

Sources: [paper](https://doi.org/10.1093/nsr/nwag004), Supplementary Table S2,
[author FigureS3.R](https://github.com/ZhengTiger/SPIDER-Seq/blob/047eb5aaa15087c241570745bd6df07882b0dd78/Supplementary%20Figures/FigureS3.R),
and [processed Adult.Ex.rds](https://huggingface.co/spaces/TigerZheng/SPIDER-web/blob/22e37e94bc5a5ecda2185f2af4eae5cf49c6092a/data/Adult.Ex.rds).

## Configuration

Run from a Python 3.10+ OnDemand kernel with the repository dependencies.
Raw and processed data, model checkpoints and results persist under BASE_DIR.
Each repetition makes new within-animal cell splits; repetitions are not
independent animals. Set RESULTS_ONLY for replotting saved results.


In [ ]:
from pathlib import Path
import os

N_OUTER_FOLDS = 3
N_REPETITIONS = 5
N_JOBS = 32
PARALLEL_UNIT = 'scenario'
USE_LOCATION = False
USE_TARGET_FEATURES = False
STRATEGY = 'full_joint'
CANDIDATE_BUDGET = 32
SEED = 20260910

# Shared expression preprocessing; every model receives exactly the same features.
# Both variable-gene selection and PCA are fitted on training cells only.
N_HVG = 2000  # Train-only variance selection from 26,902 genes; try 1000/5000 as a sensitivity check.
N_GENE_COMPONENTS = 50  # Train-only PCA; None uses N_HVG genes directly and is much slower.
LOCATION_FEATURES_CSV = None  # No native cell location in this scRNA-seq object.
TARGET_FEATURES_CSV = None    # Outcome-independent descriptors, indexed by exact target IDs.
RUN_RANDOM_FOREST = True
RUN_QIAO = True  # Target-ID baseline unless USE_TARGET_FEATURES=True.

# Native panels only: no synthetic target blocks, positive thinning, or paired references.
SHOW_FULL_DIAGNOSTICS = False
SHOW_PROGRESS = True
PROGRESS_INTERVAL_SECONDS = 60.0
EXPORT_CELL_FORECASTS = True  # Compressed CSV means/SDs; per-fold NPZs are always retained.

BASE_DIR = Path('/home/yueyue/gene2wire').expanduser()
RAW_DATA_DIR = BASE_DIR / 'raw_data'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / 'native_panels_v2'
EXPORT_DIR = BASE_DIR / 'paper_figure_exports'
FIGURE_DIR = BASE_DIR / 'figures' / '0911' / 'SPIDER_Seq_native_panels'
CODE_CACHE_DIR = BASE_DIR / 'code'
RESULTS_ONLY = False
EXISTING_EXPORT_DIRS = {'SPIDER-Seq': None}  # Exact completed run directory.

CORE_COMMIT = '6ae4d05e58309df9bb9cb8fcfb840791ac5091fd'
EXPECTED_SOURCE_HASH = '819facc9a350d3b0ae077de0c2d95ad57ddedff6d772fd32fa3be78d82b152c6'
REPO_URL = 'https://github.com/Yue-stat/Gene2Wire.git'
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'


## Load the pinned shared core


In [ ]:
REQUIRED_MODULES = ('numpy', 'scipy', 'pandas', 'sklearn', 'joblib', 'matplotlib', 'yaml', 'rdata')

import hashlib
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile

if sys.version_info < (3, 10):
    raise RuntimeError('Select an OnDemand Python 3.10 or newer kernel.')
if not re.fullmatch(r'[0-9a-f]{40}', CORE_COMMIT):
    raise RuntimeError('This notebook needs its released 40-character CORE_COMMIT pin.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_SOURCE_HASH):
    raise RuntimeError('This notebook needs its released source checksum.')

def notebook_source_hash(package_root):
    # Same byte-level convention as experiments.protocol.source_hash().
    digest = hashlib.sha256()
    for source_path in sorted(package_root.rglob('*.py')):
        digest.update(source_path.relative_to(package_root).as_posix().encode())
        digest.update(source_path.read_bytes())
    return digest.hexdigest()

def verify_checkout(checkout, require_git_pin=True):
    package_root = checkout / 'src' / 'gene2wire'
    if not package_root.is_dir():
        raise RuntimeError(f'Missing Gene2Wire sources in {checkout}')
    if require_git_pin:
        actual_commit = subprocess.check_output(
            ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if actual_commit != CORE_COMMIT:
            raise RuntimeError(f'Cached code has commit {actual_commit}, expected {CORE_COMMIT}.')
    if notebook_source_hash(package_root) != EXPECTED_SOURCE_HASH:
        raise RuntimeError(f'Source checksum mismatch in {checkout}; use the released code.')
    return checkout

# Running from the exact local repository is supported without any network call.
CORE_CHECKOUT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    package_root = candidate / 'src' / 'gene2wire'
    if package_root.is_dir() and notebook_source_hash(package_root) == EXPECTED_SOURCE_HASH:
        CORE_CHECKOUT = verify_checkout(candidate, require_git_pin=False)
        break

if CORE_CHECKOUT is None:
    CODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached_checkout = CODE_CACHE_DIR / CORE_COMMIT
    if not cached_checkout.exists():
        stage = Path(tempfile.mkdtemp(prefix='.gene2wire-download-', dir=CODE_CACHE_DIR))
        try:
            for arguments in (
                ['git', 'init', '--quiet', str(stage)],
                ['git', '-C', str(stage), 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', str(stage), 'fetch', '--quiet', '--depth', '1', 'origin', CORE_COMMIT],
                ['git', '-C', str(stage), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'],
            ):
                subprocess.run(arguments, check=True)
            verify_checkout(stage)
            try:
                stage.rename(cached_checkout)
            except OSError:
                # Another notebook may have completed this same immutable cache.
                if not cached_checkout.exists():
                    raise
                verify_checkout(cached_checkout)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
    CORE_CHECKOUT = verify_checkout(cached_checkout)

existing = sys.modules.get('gene2wire')
if existing is not None:
    same_path = Path(existing.__file__).resolve().parent == (CORE_CHECKOUT / 'src' / 'gene2wire').resolve()
    same_source = getattr(existing, '_notebook_source_hash_0908', None) == EXPECTED_SOURCE_HASH
    if not (same_path and same_source):
        raise RuntimeError('A different or unverified gene2wire is already imported. Restart the kernel, then Run All.')

missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Use an OnDemand Python kernel containing these dependencies: '
                       + ', '.join(missing) + '. See the repository environment instructions.')
sys.path.insert(0, str(CORE_CHECKOUT / 'src'))
import gene2wire
from gene2wire.experiments.protocol import Settings, source_hash
if source_hash() != EXPECTED_SOURCE_HASH:
    raise RuntimeError('Imported code does not match the released source checksum.')
gene2wire._notebook_source_hash_0908 = EXPECTED_SOURCE_HASH
for directory in (RAW_DATA_DIR, CHECKPOINT_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'core_commit': CORE_COMMIT, 'source_hash': source_hash(),
       'imported_from': gene2wire.__file__, 'raw_cache': str(RAW_DATA_DIR),
       'checkpoints': str(CHECKPOINT_DIR), 'exports': str(EXPORT_DIR)})


## Model and evaluation settings

No independent paired reference is available here. Models therefore predict
the probability of an assay-positive call. Logistic, MIRT and Joint use the
same core estimators and tuning rules as the other notebooks. Joint uses
the native candidate budget for genuine shared-plus-specific candidates
and carries the exact selected Logistic and MIRT configurations as two
mandatory endpoints when those models are in the run. This prevents the
bounded Joint grid from silently omitting the best standalone low-rank
configuration. Endpoint fits are reused from the common caches. This
does not assume perfect biological detection. RF and Qiao use the same
observed labels and feature budget.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from gene2wire.experiments.native_panel import (
    native_panel_tables, run_native_panel_experiment, summarize_native_predictions)
from gene2wire.experiments.native_panel_plotting import plot_native_panel_results
from gene2wire.experiments.reporting import (
    configure_compact_display, configure_full_display, display_diagnostics, load_existing_exports)
from gene2wire.seeds import stable_seed

if SHOW_FULL_DIAGNOSTICS:
    configure_full_display()
else:
    configure_compact_display()
settings = Settings(
    supervision_profile='assay_only', paired_fraction=0., calibration_fractions=(),
    loss_rates=(0.,), n_outer_folds=N_OUTER_FOLDS, n_repetitions=N_REPETITIONS,
    n_jobs=N_JOBS, parallel_unit=PARALLEL_UNIT, seed=SEED,
    use_location=USE_LOCATION, use_target_features=USE_TARGET_FEATURES,
    strategy=STRATEGY, candidate_budget=CANDIDATE_BUDGET,
    run_random_forest=RUN_RANDOM_FOREST, run_qiao=RUN_QIAO,
    run_information_controls=False, run_mechanism_controls=False, run_calibration_controls=False)
print({'models': [model.name for model in settings.models()],
       'RF-observed': RUN_RANDOM_FOREST, 'Qiao': RUN_QIAO,
       'folds': N_OUTER_FOLDS, 'repetitions': N_REPETITIONS, 'N_JOBS': N_JOBS,
       'gene_features': {'training_variable_genes': N_HVG, 'training_PCs': N_GENE_COMPONENTS},
       'paired_references': 'none', 'artificial_masking': 'none'})
if RESULTS_ONLY:
    all_artifacts = load_existing_exports(EXISTING_EXPORT_DIRS, expected_labels=('SPIDER-Seq',))
    artifacts = all_artifacts['SPIDER-Seq']
    if artifacts.manifest.get('experiment') != 'native_panels':
        raise ValueError('Select the SPIDER-Seq native-panel export, not spatial/block results.')
    print('RESULTS_ONLY: saved results loaded; raw loading and fitting are skipped.')


## Kernel CPU allowance and experiment workers

CPU affinity, scheduler allocation and cgroup quota are reported separately
from running experiment workers. The pool can use at most the number of
independently scheduled fold/repetition tasks.


In [ ]:
from gene2wire.experiments.workers import NotebookWorkerStatus

worker_status = None
if RESULTS_ONLY:
    print('RESULTS_ONLY: no training workers are launched.')
else:
    worker_status = NotebookWorkerStatus(requested_workers=N_JOBS)


## Load cached RNA counts and audit the real animal panels

The raw RDS is checksum pinned. Its RNA counts and processed barcode calls
are cached in sparse, pickle-free files after the first read. Adult.Ex's
excitatory cohort is retained without selecting cells by barcode positivity.
Positive calls use the author's processed values >0; raw-count thresholds
are not applied a second time. Unmeasured targets stay outside all losses.

Variable genes and PCA are selected/fitted within the training split and
refitted on development cells after model selection. Spatial coordinates
and target descriptors require independently supplied aligned CSVs when
their switches are enabled.


In [ ]:
if not RESULTS_ONLY:
    from gene2wire.experiments.datasets.spider_seq import load_spider_seq
    dataset = load_spider_seq(
        RAW_DATA_DIR / 'SPIDER_Seq', n_hvg=N_HVG, n_gene_components=N_GENE_COMPONENTS,
        location_features_csv=LOCATION_FEATURES_CSV,
        target_features_csv=TARGET_FEATURES_CSV)
    audit = native_panel_tables(dataset)
    display(audit['native_panel'].groupby('animal', observed=True).agg(
        cells=('n_cells', 'first'), measured_targets=('measured', 'sum'),
        measured_pairs=('n_measured', 'sum'), assay_positive_pairs=('n_positive', 'sum')))
    panel = audit['native_panel'].pivot(index='animal', columns='target', values='measured')
    display(panel.astype(int))
    folds = dataset.split_builder(N_OUTER_FOLDS,
        stable_seed(SEED, 'native_panel_splits', dataset.name, 0))
    display(pd.DataFrame([{'fold': f.outer_fold, 'inner_train': len(f.train_rows),
        'validation': len(f.validation_rows), 'test': len(f.test_rows)} for f in folds]))
    print('RNA counts are used; author integrated embeddings are not predictor inputs.')
    print('W is defined by injection panels. An unmeasured entry is not a negative label.')
    print('All three animals occur in every split role; this is within-animal new-cell CV.')


## Train with native panels and repeated within-animal CV

Each cell is held out once per repetition. All models receive the same
cells, features, assay mask and validation observations. Joint searches
include the exact independently tuned direct and low-rank winners as
inherited endpoints. `CANDIDATE_BUDGET` counts only genuine Joint
candidates; the maximum Joint selection set is therefore `B + 2` when
both standalone endpoints are available. Compatible complete and partial
checkpoints resume automatically. Feature preparation and model
fitting report separately in Los Angeles time. With the defaults,
5 repetitions x 3 folds x 2 fit roles gives 30 feature sets before the
model-unit progress denominator begins.


In [ ]:
if not RESULTS_ONLY:
    print('Preparing training-only features and running the shared native-panel benchmark...')
    artifacts = run_native_panel_experiment(
        dataset, settings, checkpoint_dir=CHECKPOINT_DIR, export_dir=EXPORT_DIR,
        progress=SHOW_PROGRESS, progress_interval=PROGRESS_INTERVAL_SECONDS,
        worker_status=worker_status, export_cell_forecasts=EXPORT_CELL_FORECASTS)
    all_artifacts = {'SPIDER-Seq': artifacts}


## Audit the Joint candidate budget and inherited endpoints

This compact audit is useful for interpreting a Joint result. The native
candidate budget is the number of newly evaluated genuine Joint fits;
`inherited_endpoint` rows are exact standalone winners carried into the
same validation comparison and normally loaded from the shared candidate
cache. Complete trial records remain in `tuning.csv`.


In [ ]:
if not RESULTS_ONLY:
    tuning_audit = artifacts.tables.get('tuning', pd.DataFrame()).copy()
    if not tuning_audit.empty:
        joint_audit = tuning_audit.loc[tuning_audit['model'].eq('Joint')].copy()
        if not joint_audit.empty:
            endpoint_summary = (joint_audit.assign(
                endpoint=joint_audit['stage'].eq('inherited_endpoint'),
                genuine_joint=joint_audit['kind'].eq('joint'))
                .groupby(['repetition', 'outer_fold'], observed=True)
                [['endpoint', 'genuine_joint']].sum().reset_index())
            endpoint_summary['max_selectable'] = endpoint_summary['endpoint'] + endpoint_summary['genuine_joint']
            display(endpoint_summary)
            print({'native_joint_budget': CANDIDATE_BUDGET,
                   'expected_inherited_endpoints': 2,
                   'expected_max_selectable': CANDIDATE_BUDGET + 2})
    else:
        print('No tuning table available in RESULTS_ONLY mode; inspect tuning.csv.')


## Recover observed-panel summaries and unassayed forecasts

This step can run from saved predictions alone. Measured-test metrics and
per-animal/target scores use W=1 only. W=0 forecasts carry missing reference
labels and evaluation_eligible=False; their variation across repeated splits
is not a prediction interval. Compressed per-cell CSVs summarize repeated
out-of-fold forecasts; every original fold prediction remains in units/.


In [ ]:
if RESULTS_ONLY:
    artifacts = summarize_native_predictions(artifacts, export_cell_forecasts=EXPORT_CELL_FORECASTS)
print('Run exports:', artifacts.export_dir)
print('Unassayed forecasts:', artifacts.export_dir / 'native_unassayed')


## Display figures and save PDF files

The measured-panel figures report actual held-out assay outcomes. The
unassayed heatmaps are prediction-only summaries with no reference validation.
No artificial loss-rate axis or full-panel control is introduced.


In [ ]:
figure_paths = plot_native_panel_results(artifacts, output_dir=FIGURE_DIR, show=True)
display(figure_paths)


## Essential metrics, selected hyperparameters and convergence

Compact diagnostics retain aggregate and per-animal metrics, important model
selections by repetition/fold and validation-selection evidence. Complete
metrics, per-target values, tuning records and forecasts stay in the export
directory. SHOW_FULL_DIAGNOSTICS=True opts into much larger output.


In [ ]:
display_diagnostics(artifacts, label='SPIDER-Seq native panels', full=SHOW_FULL_DIAGNOSTICS)
print('PDF figures:', FIGURE_DIR)
print('Raw and processed cache:', RAW_DATA_DIR / 'SPIDER_Seq')
print('Resumable checkpoints:', CHECKPOINT_DIR)
